In [1]:
import numpy as np
import pandas as pd

In [2]:
from surprise import Dataset, Reader, SVD

In [3]:
path = "movie_data/"

In [4]:
df_movies = pd.read_csv(path + "movies.csv")
df_ratings = pd.read_csv(path + "ratings.csv")

In [6]:
df_movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [16]:
df_ratings[(df_ratings['userId'] == 1) & (df_ratings['movieId'] == 29)]

,userId,movieId,rating,timestamp


In [17]:
algorithm = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02)

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(df_ratings[["userId", "movieId", "rating"]], reader)
trainset = data.build_full_trainset()
algorithm.fit(trainset)

In [18]:
algorithm.predict(1, 1203)

Prediction(uid=1, iid=1203, r_ui=None, est=4.879837559267652, details={'was_impossible': False})

In [19]:
def recommendation(user_id, num_recommendations):
    # Get a list of all movie IDs
    all_movie_ids = set(df_movies['movieId'])
    
    # Get the movies the user has already rated
    rated_movie_ids = set(df_ratings[df_ratings['userId'] == user_id]['movieId'])
    
    # Get the movies the user has not rated
    unrated_movie_ids = all_movie_ids - rated_movie_ids
    
    # Predict ratings for the unrated movies
    predictions = []
    for movie_id in unrated_movie_ids:
        pred = algorithm.predict(user_id, movie_id)
        predictions.append((movie_id, pred.est))
    
    # Sort the predictions by estimated rating in descending order
    predictions.sort(key=lambda x: x[1], reverse=True)
    
    # Get the top N recommendations
    top_recommendations = predictions[:num_recommendations]
    
    # Retrieve movie titles for the recommended movie IDs
    recommended_movies = []
    for movie_id, est_rating in top_recommendations:
        title = df_movies[df_movies['movieId'] == movie_id]['title'].values[0]
        recommended_movies.append((title, est_rating))
    
    return recommended_movies

In [20]:
recommendations = recommendation(1, 15)
recommendations

[('Shawshank Redemption, The (1994)', 5.0),
 ('Trainspotting (1996)', 5.0),
 ('His Girl Friday (1940)', 5.0),
 ('Streetcar Named Desire, A (1951)', 5.0),
 ('Lawrence of Arabia (1962)', 5.0),
 ("Once Upon a Time in the West (C'era una volta il West) (1968)", 5.0),
 ('Life Is Beautiful (La Vita è bella) (1997)', 5.0),
 ("Guess Who's Coming to Dinner (1967)", 5.0),
 ('Beautiful Mind, A (2001)', 5.0),
 ('Top Secret! (1984)', 5.0),
 ('American Splendor (2003)', 5.0),
 ('Dogville (2003)', 5.0),
 ('There Will Be Blood (2007)', 5.0),
 ('Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)',
  4.998895762541003),
 ("Monty Python's And Now for Something Completely Different (1971)",
  4.996171012527069)]